# 03 - Feature selection e explicabilidade

Objetivo: entender quais variaveis mais influenciam o modelo. A selecao deve ser feita com cuidado para nao usar o teste como parte da decisao.


In [ ]:
# Load libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import GradientBoostingClassifier

RANDOM_STATE = 42
TARGET = 'Churn'
ID_COLS = ['customerID']


In [ ]:
# Prepare the same split used in the project
df = pd.read_csv('../data/raw/telco_churn_raw.csv')
df['Churn'] = df['Churn'].map({'Yes': 1, 'No': 0}).astype(int)
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'].replace(' ', np.nan), errors='coerce')
X = df.drop(columns=ID_COLS + [TARGET])
y = df[TARGET]
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=RANDOM_STATE, stratify=y)


In [ ]:
# Fit pipeline and inspect feature importance
num_cols = X_train.select_dtypes(include=['number', 'bool']).columns.tolist()
cat_cols = X_train.select_dtypes(exclude=['number', 'bool']).columns.tolist()
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)
preprocess = ColumnTransformer([
    ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler())]), num_cols),
    ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('onehot', encoder)]), cat_cols),
])
model = GradientBoostingClassifier(n_estimators=150, learning_rate=0.04, max_depth=2, min_samples_leaf=25, subsample=0.85, random_state=RANDOM_STATE)
pipe = Pipeline([('preprocess', preprocess), ('model', model)])
pipe.fit(X_train, y_train)
feature_names = pipe.named_steps['preprocess'].get_feature_names_out()
importance = pd.DataFrame({
    'feature': feature_names,
    'importancia': pipe.named_steps['model'].feature_importances_
}).sort_values('importancia', ascending=False)
importance.head(20)


In [ ]:
# Plot top features
top = importance.head(15).iloc[::-1]
ax = top.plot(kind='barh', x='feature', y='importancia', legend=False, figsize=(9, 6), color='#6b8f3a')
ax.set_title('Top variaveis')
ax.set_xlabel('Importancia')
plt.tight_layout()
plt.show()
